In [1]:
import pandas as pd
import numpy as np

In [5]:
debt_df = pd.read_csv('Data/african_debt_data.csv')

print(debt_df.head())
print(debt_df.columns)

  ISO3  Country BorrowerType         BorrowerAgency              CreditorName  \
0  DZA  Algeria          NaN  Government of Algeria         World Bank (IBRD)   
1  DZA  Algeria          NaN                    NaN  African Development Bank   
2  DZA  Algeria          NaN                    NaN  Islamic Development Bank   
3  DZA  Algeria          NaN  Government of Algeria         World Bank (IBRD)   
4  DZA  Algeria          NaN  Government of Algeria         World Bank (IBRD)   

  CreditorName_short CreditorGroup             CreditorAgency  \
0            WB-IBRD  Multilateral                       IBRD   
1               AfDB  Multilateral      AfDB Ordinary Capital   
2               IsDB  Multilateral  Islamic Development Bank    
3            WB-IBRD  Multilateral                       IBRD   
4            WB-IBRD  Multilateral                       IBRD   

              CreditorAgencyType    year  ... interest  real_interest  price  \
0  Multilateral development bank  2000.0  

In [6]:
crises_df = pd.read_csv('Data/african_crises.csv')
print(crises_df.head())
print(crises_df.columns)


   case  cc3  country  year  systemic_crisis  exch_usd  \
0     1  DZA  Algeria  1870                1  0.052264   
1     1  DZA  Algeria  1871                0  0.052798   
2     1  DZA  Algeria  1872                0  0.052274   
3     1  DZA  Algeria  1873                0  0.051680   
4     1  DZA  Algeria  1874                0  0.051308   

   domestic_debt_in_default  sovereign_external_debt_default  \
0                         0                                0   
1                         0                                0   
2                         0                                0   
3                         0                                0   
4                         0                                0   

   gdp_weighted_default  inflation_annual_cpi  independence  currency_crises  \
0                   0.0              3.441456             0                0   
1                   0.0             14.149140             0                0   
2                   0.0   

In [ ]:
crises_countries = crises_df['country'].unique()
debt_countries = debt_df['Country'].unique()


common_countries = set(crises_countries) & set(debt_countries)
print(f"Common countries: {common_countries}")


Common countries: {'South Africa', 'Zimbabwe', 'Egypt', 'Mauritius', 'Nigeria', 'Algeria', 'Zambia', 'Central African Republic', 'Kenya', 'Morocco', 'Angola', 'Tunisia'}


In [13]:
# Checking the creditor types whether it is domestic or externaly.
print(debt_df['CreditorName'].value_counts())

# Create classification (I have defined these based on the data)
domestic_creditors = ['Domestic Bank', 'Central Bank', 'Local']  
external_creditors = ['WB-IBRD', 'AfDB', 'IsDB', 'IMF', 'China', 'France']  

debt_df['debt_type'] = debt_df['CreditorName_short'].apply(
    lambda x: 'Domestic' if x in domestic_creditors else 'External'
)

CreditorName
Domestic Markets                                    47583
World Bank (IDA)                                     3183
China                                                1468
France                                                663
African Development Fund                              544
Islamic Development Bank                              501
OPEC Fund for International Development               348
International Fund for Agricultural Development       295
World Bank (IBRD)                                     288
Germany                                               277
African Development Bank                              265
Japan                                                 215
Bondholders                                           200
International Monetary Fund                           200
Arab Bank for Economic Development in Africa          118
Kuwait                                                111
International Finance Corporation                      92
A

In [11]:
# Filter for common countries and 2000-2014
debt_filtered = debt_df[
    (debt_df['Country'].isin(common_countries)) & 
    (debt_df['year'] >= 2000) & 
    (debt_df['year'] <= 2014)
]

# Aggregating by country, year, and debt type
debt_annual = debt_filtered.groupby(['Country', 'year', 'debt_type']).agg({
    'Amount_musd': 'sum',
    'maturity': 'mean',
    'interest': 'mean'
}).reset_index()

# Pivot to get domestic and external as columns
debt_pivot = debt_annual.pivot_table(
    index=['Country', 'year'],
    columns='debt_type',
    values='Amount_musd',
    aggfunc='sum',
    fill_value=0
).reset_index()

print(debt_pivot.head())


debt_type  Country    year  External
0          Algeria  2000.0     301.0
1          Algeria  2001.0     148.0
2          Algeria  2002.0     251.0
3          Algeria  2003.0     319.0
4          Algeria  2004.0      62.0


KeyError: 'domestic_debt_musd'